### STEP1
✔  Extract text
✔  Clean text
✔  Chunk text
✔  Add metadata
✔  Create dataset

###                                  RAG-BASED POLICY QUESTION-ANSWERING PROJECT

In [104]:
!pip install PyPDF2

import os
import re
import PyPDF2
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

# STEP 1: EXTRACT TEXT FROM PDF

def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "
    return text


# STEP 2: CLEAN TEXT

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)

    # remove special characters
    text = re.sub(r'[^a-zA-Z0-9.,;:()\-\/ ]', '', text)

    # remove section numbers like (1), (2)
    text = re.sub(r'\(\d+\)', '', text)

    return text.strip()


# STEP 3: CHUNK TEXT (100\u2013500 WORDS)

def chunk_text(text, max_words=120):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        word_count = len(sentence.split())

        if current_length + word_count <= max_words:
            current_chunk.append(sentence)
            current_length += word_count
        else:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


# STEP 4 + 5: PROCESS ALL DOCUMENTS + STORE METADATA

def process_documents(folder_path):
    dataset = []

    for file_name in os.listdir(folder_path):

        if file_name.endswith(".pdf"):

            file_path = os.path.join(folder_path, file_name)

            print(f"Processing: {file_name}")

            # STEP 1: extract
            raw_text = extract_text_from_pdf(file_path)

            # STEP 2: clean
            cleaned_text = clean_text(raw_text)

            # STEP 3: chunk
            chunks = chunk_text(cleaned_text, max_words=120)

            # STEP 4: store metadata
            for idx, chunk in enumerate(chunks):

                dataset.append({
                    "chunk_id": f"{file_name}_chunk_{idx}",
                    "source_document": file_name,
                    "chunk_index": idx,
                    "text": chunk,
                    "word_count": len(chunk.split())
                })

    return dataset



# RUN PIPELINE

folder_path = "data"   # change to your folder path

dataset = process_documents(folder_path)


# STEP 5: CONVERT TO DATAFRAME (FINAL DATASET)

df = pd.DataFrame(dataset)


# Save dataset
df.to_csv("policy_chunks_dataset.csv", index=False)

print("\nDataset created successfully!")
print(f"Total chunks: {len(df)}")

df.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Processing: FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf
Processing: Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf
Processing: Makerere-Policy-on-Persons-Living-With-Disabilities.pdf
Processing: HIV_AIDS_Policy.pdf
Processing: Makerere-Safeguarding-Policy.pdf

Dataset created successfully!
Total chunks: 271


,chunk_id,source_document,chunk_index,text,word_count
0,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf...,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf,0,"1 GOVERNMENT OF UGANDA MINISTRY OF GENDER, LAB...",39
1,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf...,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf,1,"Box 7136, Kampala - Uganda Website: : http//ww...",428
2,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf...,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf,2,ROLES AND RESPONSIBILITIES OF STAKEHOLDERS ......,110
3,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf...,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf,3,The Policy has been reviewed following an earl...,100
4,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf...,FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf,4,The revision was also premised on dynamism in ...,117


In [105]:
pip install pyspellchecker

In [106]:
from spellchecker import SpellChecker

# Initialize spell checker
spell = SpellChecker()

# STEP 2: CLEAN TEXT (Enhanced with spell checking)
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)

    # remove special characters, keeping only letters, numbers, and some punctuation
    text = re.sub(r'[^a-zA-Z0-9.,;:()\-\/ ]', '', text)

    # remove section numbers like (1), (2)
    text = re.sub(r'\(\d+\)', '', text)

    # Split text into words, correct them, and join back
    words = text.split()
    corrected_words = [spell.correction(word) if spell.correction(word) is not None else word for word in words]

    # Remove 'None' values if any occurred during correction (shouldn't with the None check, but as a safeguard)
    corrected_words = [word for word in corrected_words if word is not None]

    return ' '.join(corrected_words).strip()

In [107]:
import os

# Define the folder path
folder_path = "data"

# Create the folder if it doesn't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_path}' created successfully.")
else:
    print(f"Folder '{folder_path}' already exists.")

Folder 'data' already exists.


### STEP2:
Convert chunks into vectors (EMBEDDINGS)

In [116]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Ensure all text entries are strings before encoding
texts = df['text'].astype(str).tolist()

# encode safely (better for large datasets)
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True   # IMPORTANT FIX
)

# save embeddings
np.save("chunk_embeddings.npy", embeddings)

# safety check
df.to_csv("chunk_metadata.csv", index=False)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [115]:
import re
from sklearn.metrics.pairwise import cosine_similarity

def keyword_filter(df, query, column='text'):
    # Basic keyword extraction (can be improved with NLP libraries)
    keywords = [word.lower() for word in re.findall(r'\b\w+\b', query) if len(word) > 2]

    if not keywords:
        return df  # Return original DataFrame if no keywords are found

    # Filter chunks that contain at least one keyword
    filtered_df = df[df[column].apply(lambda x: any(keyword in str(x).lower() for keyword in keywords))]
    return filtered_df

def retrieve_top_k(query, model, embeddings, df, k=5, threshold=0.3):

    # 1. keyword filter FIRST
    filtered_df = keyword_filter(df, query)

    if len(filtered_df) == 0:
        filtered_df = df  # fallback

    # align embeddings
    filtered_indices = filtered_df.index.tolist()
    filtered_embeddings = embeddings[filtered_indices]

    # 2. similarity search
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, filtered_embeddings)[0]

    sorted_idx = np.argsort(similarities)[::-1]

    top_idx = sorted_idx[:k]

    results = filtered_df.iloc[top_idx].copy()
    results["similarity_score"] = similarities[top_idx]

    return results

## step: 3
##ACTUAL RAG (Retrieval Augmented Generation)

In [110]:
#load both dataset and embeddings for future use
df = pd.read_csv("policy_chunks_dataset.csv")
embeddings = np.load("chunk_embeddings.npy")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize the text generation model
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    clean_up_tokenization_spaces=True,
    device=-1

)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

## RAG Answer Generation

In [119]:
def generate_answer(query, retrieved):
    import re

    # 🔧 Clean messy OCR text
    def clean_text(text):
        text = re.sub(r'\s+', ' ', text)

        # Fix common broken words from your data
        text = text.replace("functi on", "function")
        text = text.replace("vi on", "vision")
        text = text.replace("do- able", "doable")

        return text

    # 🔽 Use fewer chunks (reduces confusion)
    top_chunks = retrieved.head(2)

    if top_chunks.empty:
        return "No relevant information found."

    # 🔧 Apply cleaning
    context = "\n\n".join([clean_text(t) for t in top_chunks['text']])

    # 🔧 Simpler, stronger prompt
    prompt = f"""
Context:
{context}

Question:
{query}

Answer in clear bullet points:
"""

    # 🔧 Generate answer
    result = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )



    answer = result[0]['generated_text'].strip()

    # ❗ TEMP: do NOT block short answers yet
    return answer

## FULL RAG PIPELINE FUNCTION

In [122]:
def rag_answer(query):

    retrieved = retrieve_top_k(query, embedding_model, embeddings, df, k=5)

    answer = generate_answer(query, retrieved)

    print("\n✅ Question:", query)
    print("\n❑ Answer:\n", answer)

    print("\n✅ Sources:")
    for src in retrieved['source_document'].unique():
        print("-", src)


## TEST YOUR RAG SYSTEM

In [123]:
rag_answer("how to file a complaint ?")

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Question: how to file a complaint ?

❑ Answer:
 Context:
11.1 Informal complaint (a) An inf ormal complaint is made to any member of the implementing bodies, including academic staff, administrative staff or students leader. 11It is made in those cases where the victim wishes for immediate action to be taken (for instance, warn the harasser or change his/her dissertation supervisor) to remedy or redress the harm without pursuing disciplinary action or seeking sanctions against the respondent . In any case, the respondent must be notified of the complaint lodged against him/her. (b) An informal complaint lodged with a student leader, academic or administrative staff shall be forwarded to the Gender Mainstreaming Directorate for recording.

(b) Wher e there are allegations of conflict of interest by a member, the latter shall recuse him/herself from the Committee investigating the alleged sexual harassment. 19. Lodging of a complaint (a) A complaint of sexual harassment should be lodg

In [117]:
!pip install gradio

In [124]:
import gradio as gr

def gradio_rag_interface(query):
    # The rag_answer function already prints the output,
    # but for Gradio, we need to capture it or modify it to return strings.
    # Let's modify rag_answer slightly to return the answer and sources.

    # Temporarily redirect stdout to capture print statements
    import io
    import sys
    old_stdout = sys.stdout
    redirected_output = io.StringIO()
    sys.stdout = redirected_output

    rag_answer(query)

    sys.stdout = old_stdout # Restore stdout
    output_string = redirected_output.getvalue()

    # Parse the output string to separate answer and sources if needed for better display
    # For simplicity, we'll return the full output for now.
    return output_string

# Create the Gradio interface
iface = gr.Interface(fn=gradio_rag_interface,
                     inputs=gr.Textbox(lines=2, placeholder="Ask a question about the policy documents..."),
                     outputs=gr.Textbox(lines=15, label="Answer"), # Increased lines for output
                     title="Policy Question-Answering System",
                     description="Ask questions related to the uploaded policy documents. The system will retrieve relevant information and generate an answer.")

# Launch the Gradio interface
# Set share=True to get a public link, useful for sharing or accessing on other devices
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73080fdd75cba14351.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Confirming All Processed Documents

In [114]:
# Display all unique source documents in the DataFrame to confirm all policies were processed
print("Unique Source Documents Processed:")
for doc in df['source_document'].unique():
    print(f"- {doc}")


Unique Source Documents Processed:
- FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf
- Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf
- HIV_AIDS_Policy.pdf


## QNS: 2

## QNS: 3

## QNS: 4